Load and verify all Gold datasets

In [0]:

from pyspark.sql import functions as F

# Step 1: Define Gold storage location
gold_base_path = "/Volumes/workspace/default/railway_gold"

# Step 2: Define Gold datasets without hardcoded counts
gold_datasets = [
    "overall_kpis",
    "station_performance",
    "train_performance",
    "monthly_performance",
    "yearly_performance",
    "station_recovery"
]

# Step 3: Store loaded DataFrames
gold_dataframes = {}

# Step 4: Read and validate each dataset
for dataset_name in gold_datasets:

    dataset_path = f"{gold_base_path}/{dataset_name}"

    # Read current Gold Delta dataset
    df = spark.read.format("delta").load(dataset_path)

    # Calculate current metrics
    actual_count = df.count()
    column_count = len(df.columns)

    # Validate dataset is not empty
    assert actual_count > 0, (
        f"{dataset_name}: Dataset is empty!"
    )

    # Validate schema is not empty
    assert column_count > 0, (
        f"{dataset_name}: No columns found!"
    )

    # Validate overall KPI grain
    if dataset_name == "overall_kpis":
        assert actual_count == 1, (
            f"overall_kpis must contain 1 row, "
            f"found {actual_count}"
        )

    # Store for downstream export
    gold_dataframes[dataset_name] = df

    print(
        f"{dataset_name} | "
        f"Records: {actual_count} | "
        f"Columns: {column_count} | "
        f"Status: PASSED"
    )

print("\nALL GOLD DATASETS LOADED SUCCESSFULLY")


In [0]:

station_df = gold_dataframes["station_performance"]

# Check available columns
print("Station Performance Columns:")
print(station_df.columns)

# Check total station rows
print("Total Station Rows:", station_df.count())

# Check unique station codes
unique_stations = station_df.select(
    "station_code"
).distinct().count()

print("Unique Station Codes:", unique_stations)

# Check duplicate station codes
duplicate_stations = (
    station_df
    .groupBy("station_code")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate Station Groups:", duplicate_stations.count())

display(duplicate_stations)


Export Gold datasets to CSV

In [0]:
#  CSV export base directory
export_base_path = (
    "/Volumes/workspace/default/"
    "railway_gold/exports"
)

# Track exported datasets
export_results = []

# Export each Gold DataFrame
for dataset_name, df in gold_dataframes.items():

    # Define output directory
    export_path = f"{export_base_path}/{dataset_name}"

    # Export DataFrame as CSV
    # coalesce(1) creates one CSV part file per dataset
    # header=True preserves column names
    # overwrite makes exports repeatable
    (
        df.coalesce(1)
          .write
          .format("csv")
          .option("header", "true")
          .mode("overwrite")
          .save(export_path)
    )

    # Verify that a CSV part file was created
    csv_files = [
        file.path
        for file in dbutils.fs.ls(export_path)
        if file.name.startswith("part-")
        and file.name.endswith(".csv")
    ]

    # Validate that exactly one CSV part file exists
    assert len(csv_files) == 1, (
        f"{dataset_name}: Expected 1 CSV file, "
        f"found {len(csv_files)}"
    )

    # Save export information
    export_results.append(
        (dataset_name, df.count(), csv_files[0])
    )

    # Display export confirmation
    print(f"EXPORTED: {dataset_name}")
    print(f"Records: {df.count()}")
    print(f"CSV File: {csv_files[0]}")
    print("-" * 50)

# Final confirmation
print("\nALL GOLD DATASETS EXPORTED SUCCESSFULLY")
print("Total Datasets:", len(export_results))

Validate exported CSV files

In [0]:

# Track validation results
validation_results = []

# Validate each exported dataset
for dataset_name, gold_df in gold_dataframes.items():

    # Construct CSV export directory
    csv_path = f"{export_base_path}/{dataset_name}"

    # Read exported CSV
    # Keep values as strings for this structural validation
    csv_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .load(csv_path)
    )

    # Count records in Gold Delta and CSV
    gold_count = gold_df.count()
    csv_count = csv_df.count()

    # Count columns in Gold Delta and CSV
    gold_columns = len(gold_df.columns)
    csv_columns = len(csv_df.columns)

    # Validate row counts
    assert gold_count == csv_count, (
        f"{dataset_name}: Row count mismatch! "
        f"Gold={gold_count}, CSV={csv_count}"
    )

    # Validate column counts
    assert gold_columns == csv_columns, (
        f"{dataset_name}: Column count mismatch! "
        f"Gold={gold_columns}, CSV={csv_columns}"
    )

    # Validate column names and their order
    assert gold_df.columns == csv_df.columns, (
        f"{dataset_name}: Column names do not match!"
    )

    # Store validation results
    validation_results.append({
        "dataset": dataset_name,
        "gold_records": gold_count,
        "csv_records": csv_count,
        "columns": csv_columns,
        "status": "PASSED"
    })

    # Print individual dataset validation
    print(f"Dataset: {dataset_name}")
    print(f"Gold Records: {gold_count}")
    print(f"CSV Records: {csv_count}")
    print(f"Columns: {csv_columns}")
    print("Status: PASSED")
    print("-" * 50)

# Display validation summary
validation_df = spark.createDataFrame(validation_results)

display(validation_df)

# Final confirmation
print("\nALL CSV EXPORT VALIDATIONS PASSED")
print("Total Datasets Validated:", len(validation_results))